# Name That Model - Training Pipeline

Trains a classifier to identify AI language model authorship from text passages.

In [1]:
import pandas as pd
import numpy as np
import re
import html
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

Libraries loaded


## 1. Load and Preprocess Data

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f'Training samples: {len(train)}')
print(f'Test samples: {len(test)}')

def normalize_text(t):
    if pd.isna(t):
        return ''
    t = str(t)
    t = html.unescape(t)
    t = t.replace('\u2019', "'").replace('\u2018', "'").replace('\u201c', '"').replace('\u201d', '"')
    t = t.replace('\u2013', '-').replace('\u2014', '--').replace('\u2026', '...')
    t = t.replace('\ufffd', '')
    t = re.sub(r'[\r\n\t]+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

train['text_norm'] = train['text'].apply(normalize_text)
test['text_norm'] = test['text'].apply(normalize_text)

if 'model' in train.columns:
    print(f'Number of classes: {train["model"].nunique()}')
    print(train['model'].value_counts())
else:
    print('ERROR: model column not found in train.csv')

print('Data preprocessing complete')

Training samples: 2699
Test samples: 3
Number of classes: 18
model
qwen3.8-max               150
claude-sonnet-5           150
gemini-3.6-flash          150
gpt-5.6-terra             150
glm-5.3-flash             150
gemini-3.1-pro-preview    150
gemini-3.8-flash          150
claude-opus-4-6           150
gpt-5.6-luna              150
claude-opus-4-8           150
gemini-3.7-flash          150
claude-opus-5             150
kimi-k3                   150
gpt-5.5                   150
gpt-6-astra               150
glm-5.3                   150
gpt-5.6-sol               150
claude-fable-5            149
Name: count, dtype: int64
Data preprocessing complete


## 2. Build Pipeline with TF-IDF Features

In [3]:
# Text preprocessing pipeline
char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(2, 4),
    max_features=5000,
    lowercase=False,
    sublinear_tf=True
)

word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    max_features=5000,
    stop_words=None,
    lowercase=True,
    sublinear_tf=True,
    min_df=2,
    max_df=0.9
)

from sklearn.pipeline import Pipeline, FeatureUnion
text_features = FeatureUnion([
    ('char', char_vectorizer),
    ('word', word_vectorizer)
])

# Full pipeline
pipeline = Pipeline([
    ('tfidf', text_features),
    ('clf', LogisticRegression(
        max_iter=2000,
        solver='lbfgs',
        C=0.5,
        class_weight='balanced',
        random_state=42
    ))
])

print('Pipeline defined')

Pipeline defined


## 3. Train Model

In [4]:
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(train['model'])

print('Training classifier...')
pipeline.fit(train['text_norm'].astype(str), encoded_labels)
print('Training complete')

Training classifier...


Training complete


## 4. Evaluate

In [5]:
train_preds = pipeline.predict(train['text_norm'].astype(str))
bal_acc = balanced_accuracy_score(encoded_labels, train_preds)
acc = accuracy_score(encoded_labels, train_preds)
print(f'Accuracy: {acc:.4f}')
print(f'Balanced accuracy: {bal_acc:.4f}')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, train['text_norm'].astype(str), encoded_labels, cv=cv, scoring='balanced_accuracy')
print(f'Cross-validated balanced accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

Accuracy: 0.9526
Balanced accuracy: 0.9526


Cross-validated balanced accuracy: 0.1293 (+/- 0.0137)


## 5. Predict and Create Submission

In [6]:
test_preds = pipeline.predict(test['text_norm'].astype(str))
test_labels = label_encoder.inverse_transform(test_preds)

valid_models = [
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'
]
submission = pd.DataFrame({
    'id': test['id'].astype(str),
    'model': test_labels
})
submission['model'] = submission['model'].apply(lambda x: x if x in valid_models else 'gpt-5.5')
submission = submission.drop_duplicates(subset=['id'], keep='first')
submission = submission.dropna(subset=['id', 'model'])
submission.to_parquet('submission.parquet', index=False)
print('Submission created: submission.parquet')
print(submission.shape)
print(submission.to_string(index=False))

Submission created: submission.parquet
(3, 2)
          id                  model
00353fb1c541       gemini-3.8-flash
005f5a9a6139 gemini-3.1-pro-preview
0062a53d5487       gemini-3.7-flash
